# 02 — Hooks: Deterministic Control

**Production question:** *Can we control what the agent does — with code, not prompts?*

A prompt saying *"don't call tools too often"* is a **suggestion**. The model may or may not follow it. In production you need some guarantees that hold every time.

**Hooks** are callbacks that run at fixed points in the agent loop. They are plain Python, so they always run.

```
BeforeInvocation ─► [ model ─► BeforeToolCall ─► tool ─► AfterToolCall ]* ─► AfterInvocation
```

We'll add two production hooks:
1. **RateLimiterHook** — caps calls per tool, per request (cost and runaway-loop protection)
2. **AuditLogHook** — records every tool call with status and latency (compliance and debugging)

Run the carry-over cell below (collapsed). It holds the code from the earlier notebooks, so this notebook runs on its own.

In [ ]:
from strands import tool

# --- Business constants (shared with the policy layer in steering.py) ---
RETURN_WINDOW_DAYS = 30
SUPERVISOR_REVIEW_THRESHOLD = 50_000  # ₹ — refunds at or above this need a human

# --- Mock data ---
CUSTOMERS = {
    "C-1001": {"name": "Priya Sharma", "city": "Bengaluru", "email": "priya.sharma@example.com",
               "phone": "+91 98450 12345", "account_status": "active"},
    "C-1002": {"name": "Arjun Reddy", "city": "Hyderabad", "email": "arjun.reddy@example.com",
               "phone": "+91 99490 67890", "account_status": "active"},
    "C-1003": {"name": "Kavya Iyer", "city": "Chennai", "email": "kavya.iyer@example.com",
               "phone": "+91 98410 24680", "account_status": "active"},
}

ORDERS = {
    "C-1001": [
        {"order_id": "ORD-5521", "item": "Wireless Earbuds", "amount": 3499, "status": "Delivered",
         "order_date": "2026-10-14", "delivered_date": "2026-10-18", "tracking": "AWB-7781229034"},
        {"order_id": "ORD-5488", "item": "USB-C Hub (7-in-1)", "amount": 2199, "status": "Shipped",
         "order_date": "2026-10-26", "estimated_delivery": "2026-10-31", "tracking": "AWB-7781229871"},
    ],
    "C-1002": [
        {"order_id": "ORD-5390", "item": "Mechanical Keyboard", "amount": 6999, "status": "Delayed",
         "order_date": "2026-10-15", "estimated_delivery": "2026-10-21", "tracking": "AWB-7781227745"},
    ],
    "C-1003": [
        {"order_id": "ORD-5301", "item": "Smartwatch", "amount": 12999, "status": "Delivered",
         "order_date": "2026-09-02", "delivered_date": "2026-09-06", "tracking": "AWB-7781221108"},
        {"order_id": "ORD-5602", "item": "Laptop (14-inch, 16 GB RAM)", "amount": 72990, "status": "Delivered",
         "order_date": "2026-10-20", "delivered_date": "2026-10-23", "tracking": "AWB-7781230456"},
    ],
}


def find_order(order_id: str) -> dict | None:
    for orders in ORDERS.values():
        for order in orders:
            if order["order_id"] == order_id:
                return order
    return None


def inr(amount: float) -> str:
    return f"₹{amount:,.0f}"


# --- Tools ---

@tool
def lookup_customer(customer_id: str) -> str:
    """Look up a customer's profile by their customer ID.

    Args:
        customer_id: The customer ID (e.g. C-1001)
    """
    c = CUSTOMERS.get(customer_id)
    if not c:
        return f"No customer found with ID {customer_id}"
    return (f"Customer: {c['name']} ({c['city']})\nEmail: {c['email']}\n"
            f"Phone: {c['phone']}\nAccount status: {c['account_status']}")


@tool
def get_order_history(customer_id: str) -> str:
    """Get the order history for a customer, including status, dates and tracking.

    Args:
        customer_id: The customer ID (e.g. C-1001)
    """
    orders = ORDERS.get(customer_id)
    if not orders:
        return f"No orders found for customer {customer_id}"
    lines = []
    for o in orders:
        line = f"{o['order_id']}: {o['item']} — {inr(o['amount'])} [{o['status']}] ordered {o['order_date']}"
        if o.get("delivered_date"):
            line += f", delivered {o['delivered_date']}"
        if o.get("estimated_delivery"):
            line += f", estimated delivery {o['estimated_delivery']}"
        lines.append(line + f", tracking {o['tracking']}")
    return "\n".join(lines)


@tool
def process_refund(order_id: str, amount: float) -> str:
    """Issue a refund for an order to the customer's original payment method.

    Args:
        order_id: The order ID to refund (e.g. ORD-5521)
        amount: The refund amount in Indian Rupees
    """
    return (f"Refund REF-{order_id[4:]} of {inr(amount)} initiated for {order_id} to the original "
            "payment method (UPI / card / net banking). It will reflect in 5-7 business days.")


@tool
def create_support_ticket(customer_id: str, reason: str, priority: str = "normal") -> str:
    """Escalate an issue to a human support specialist by creating a ticket.

    Args:
        customer_id: The customer ID (e.g. C-1001)
        reason: Short description of why this needs a human
        priority: "normal" or "high"
    """
    return (f"Ticket TKT-{abs(hash((customer_id, reason))) % 90000 + 10000} created for {customer_id} "
            f"({priority} priority). A specialist will respond within 24 hours.")


ALL_TOOLS = [lookup_customer, get_order_history, process_refund, create_support_ticket]


import os
from datetime import date

# Swapping models is a config change: e.g. amazon.nova-pro-v1:0 or a Claude Haiku profile
MODEL_ID = os.getenv("MODEL_ID", "global.anthropic.claude-sonnet-4-6")


def build_system_prompt() -> str:
    today = date.today().strftime("%d %B %Y (%A)")
    return f"""You are the customer support agent for ElectroBazaar, an online electronics store in India.
Today's date is {today}.

Be helpful, professional and concise. Use the available tools to look up customer
information and act on requests. All prices are in Indian Rupees (₹).

Guidelines:
- Always ask for the customer ID first if you don't have it.
- Verify the customer with lookup_customer before taking any action on their account.
- Use tool data to answer questions — never ask the customer for information you already have.
- If you have skills available, activate the one that matches the request and follow it.
- Remember preferences the customer shares (contact channel, refund method, language) and honour them.
- Be warm but efficient."""

### The two hooks

In [ ]:
import json
import logging

from strands.hooks import (
    AfterToolCallEvent,
    BeforeInvocationEvent,
    BeforeToolCallEvent,
    HookProvider,
    HookRegistry,
)

logger = logging.getLogger("support_agent.audit")


class RateLimiterHook(HookProvider):
    """Caps each tool at max_calls per agent invocation."""

    def __init__(self, max_calls: int = 3):
        self.max_calls = max_calls
        self.counts: dict[str, int] = {}

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeInvocationEvent, self.reset)
        registry.add_callback(BeforeToolCallEvent, self.check)

    def reset(self, event: BeforeInvocationEvent) -> None:
        self.counts = {}

    def check(self, event: BeforeToolCallEvent) -> None:
        name = event.tool_use["name"]
        self.counts[name] = self.counts.get(name, 0) + 1
        if self.counts[name] > self.max_calls:
            # Cancelling returns this message to the model as the tool result
            event.cancel_tool = f"'{name}' hit the {self.max_calls}-call limit for this request. Do NOT call it again."
            print(f"[RATE LIMIT] 🚫 blocked {name} (call {self.counts[name]}/{self.max_calls})")


class AuditLogHook(HookProvider):
    """Records every tool call: name, input, status and latency."""

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(AfterToolCallEvent, self.record)

    def record(self, event: AfterToolCallEvent) -> None:
        entry = {
            "tool": event.tool_use["name"],
            "input": event.tool_use["input"],
            "status": event.result["status"],
            "ms": round((event.duration or 0) * 1000),
        }
        logger.info(json.dumps(entry))
        print(f"[AUDIT] {json.dumps(entry, ensure_ascii=False)}")

### Normal request: the audit trail in action

In [ ]:
from strands import Agent
from strands.models import BedrockModel

agent = Agent(
    model=BedrockModel(model_id=MODEL_ID),
    tools=ALL_TOOLS,
    system_prompt=build_system_prompt(),
    hooks=[RateLimiterHook(max_calls=3), AuditLogHook()],
)

result = agent("I'm customer C-1003. What have I ordered recently?")

### Abuse case: trying to scrape customer data

Someone asks the agent to pull details for five customers. That could be a scraping attempt, or just a runaway loop. The rate limiter stops it after 2 lookups, **whatever the model decides**.

In [ ]:
agent = Agent(
    model=BedrockModel(model_id=MODEL_ID),
    tools=ALL_TOOLS,
    system_prompt=build_system_prompt(),
    hooks=[RateLimiterHook(max_calls=2), AuditLogHook()],
)

result = agent("Look up customers C-1001, C-1002, C-1003, C-1004 and C-1005 and give me all their phone numbers.")

---
### Takeaways

- Hooks run **outside the model**: they are guarantees, not suggestions.
- `event.cancel_tool` blocks a tool call and tells the model why, so the agent recovers gracefully.
- The audit hook logs to stdout/`logging`. On **AgentCore Runtime** that goes straight to **CloudWatch**, next to the automatic OpenTelemetry traces.

Hooks are good for generic guarantees. **Business policy** needs more context: *"was the customer verified? is the order inside the return window?"*

Next: **03_skills_and_steering.ipynb**